# GetAround — analyse des retards et API de prix

## Objectif

Ce notebook présente une version lisible et publiable de l'analyse GetAround.
Il traite deux sujets complémentaires :

1. la recommandation d'un **délai minimal entre deux locations successives** ;
2. la synthèse du **modèle de prédiction du prix journalier** exposé ensuite via FastAPI.

La logique suivie est la suivante :

**problème métier → données → méthode → résultats → interprétation → limites → conclusion**


## 1. Contexte business

Le problème pertinent n'est pas simplement « certains conducteurs rendent leur voiture en retard ».
La vraie question est : **à partir de quel niveau de retard ce comportement perturbe réellement la location suivante** ?

L'analyse se concentre donc sur les locations liées à une location précédente, puis mesure l'effet de plusieurs seuils de délai minimal.
L'objectif est d'obtenir une règle opérationnelle claire, défendable et proportionnée.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

delay = pd.read_excel(ROOT / "data" / "get_around_delay_analysis.xlsx", sheet_name="rentals_data")
thresholds = pd.read_csv(ROOT / "data" / "threshold_simulation.csv")
summary = json.loads((ROOT / "data" / "business_summary.json").read_text(encoding="utf-8"))
pricing_metrics = json.loads((ROOT / "models" / "pricing_metrics.json").read_text(encoding="utf-8"))

overview = pd.DataFrame({
    "Indicateur": [
        "Locations totales",
        "Locations liées",
        "Cas problématiques",
        "Part des locations liées",
        "Part des cas problématiques dans le total",
    ],
    "Valeur": [
        summary["dataset"]["all_rentals"],
        summary["dataset"]["linked_rentals"],
        summary["operational_risk"]["problematic_cases"],
        f'{summary["dataset"]["linked_rentals_share"]*100:.1f} %',
        f'{summary["operational_risk"]["problematic_share_of_all_rentals"]*100:.1f} %',
    ],
})
overview


## 2. Méthode retenue pour l'analyse des retards

La recommandation ne repose pas sur une intuition, mais sur une logique simple :

- tester plusieurs seuils de délai minimal ;
- mesurer combien de locations seraient impactées ;
- mesurer combien de cas problématiques seraient évités ;
- retenir le **plus petit seuil** qui apporte une protection élevée sans pénaliser exagérément l'activité.

Dans ce projet, l'impact commercial est approché par le **nombre de locations affectées**, car le jeu de données ne contient pas de variable directe de revenu.


In [ ]:
scorecard = thresholds[
    (thresholds["scope"].isin(["all", "connect"]))
    & (thresholds["threshold_min"].isin([30, 60, 90, 120, 150, 180]))
][[
    "scope",
    "threshold_min",
    "affected_rentals",
    "affected_share_of_all_rentals",
    "solved_problematic_cases",
    "solved_share_of_all_problematic_cases",
]].copy()

scorecard["affected_share_of_all_rentals"] = (scorecard["affected_share_of_all_rentals"] * 100).round(1)
scorecard["solved_share_of_all_problematic_cases"] = (scorecard["solved_share_of_all_problematic_cases"] * 100).round(1)

scorecard = scorecard.rename(columns={
    "scope": "Périmètre",
    "threshold_min": "Seuil (min)",
    "affected_rentals": "Locations affectées",
    "affected_share_of_all_rentals": "Part des locations affectées (%)",
    "solved_problematic_cases": "Cas problématiques résolus",
    "solved_share_of_all_problematic_cases": "Part des cas résolus (%)",
})

scorecard


In [ ]:
display(Image(filename=ROOT / "assets" / "threshold_tradeoff.png"))

## 3. Résultat principal et interprétation

La recommandation retenue est :

- **120 minutes sur l'ensemble du parc** ;
- **180** cas problématiques résolus ;
- **82.6 %** des cas problématiques couverts ;
- **666** locations affectées ;
- **3.1 %** du total des locations affectées.

### Pourquoi cette option est retenue

Ce seuil est le premier à dépasser **80 %** de résolution des cas problématiques.
Une option plus prudente existe : **120 minutes sur Connect uniquement**.
Elle réduit davantage l'impact global, mais elle ne traite qu'une part beaucoup plus faible du problème à l'échelle de la plateforme.


## 4. Synthèse du volet machine learning

Le second volet du projet consiste à prédire un **prix journalier de location** à partir des caractéristiques du véhicule.
Le modèle retenu est ensuite exposé via FastAPI pour fournir un service simple et directement testable.


In [ ]:
metrics_table = pd.DataFrame([{
    "Modèle retenu": pricing_metrics["best_model"],
    "MAE (€/jour)": round(pricing_metrics["chosen_model_metrics"]["MAE"], 2),
    "RMSE": round(pricing_metrics["chosen_model_metrics"]["RMSE"], 2),
    "R²": round(pricing_metrics["chosen_model_metrics"]["R2"], 3),
}])

top_features = pd.DataFrame(pricing_metrics["top_features"]).copy()
top_features["importance"] = top_features["importance"].round(3)

display(metrics_table)
top_features


In [ ]:
display(Image(filename=ROOT / "assets" / "feature_importance.png"))

## 5. Exposition via API

Le dépôt contient une application **FastAPI** avec :

- `POST /predict` pour la prédiction ;
- `/docs` pour la documentation interactive ;
- `/metadata` pour les variables attendues et les métriques principales ;
- `/health` pour le contrôle simple du service.

Cette partie permet de montrer que le projet ne s'arrête pas à une analyse en notebook : le modèle est aussi présenté comme un service exploitable.


## 6. Limites du projet

- L'analyse des retards ne mesure pas directement un manque à gagner en euros.
- La recommandation est fondée sur les données fournies ; elle devrait être confirmée par un test réel en production.
- Le modèle de prix est adapté à un cas pédagogique et à une démonstration d'API, mais pas à lui seul à un usage industriel complet.
- Le dépôt conserve volontairement le fichier modèle pour permettre une exécution locale immédiate.

## Conclusion

Le projet apporte une réponse cohérente à deux questions différentes mais complémentaires :

- **une question produit** : quel délai minimal faut-il recommander entre deux locations successives ;
- **une question technique** : comment exposer un modèle de prix sous forme de service.

La recommandation issue de l'analyse est claire : **120 minutes sur l'ensemble du parc**.
Le reste du dépôt montre ensuite comment transformer cette analyse en application et en API, dans une structure sobre et directement publiable sur GitHub.
